# FacilityClient — ALCF Demo

Demonstrates the `amscrot` high-level `FacilityClient` API against the
**Argonne Leadership Computing Facility (ALCF)** IRI endpoint.

Authentication uses Globus; tokens are managed by
`scripts/tokens/get_globus_token.py`.

## What this notebook covers

| Section | Methods |
|---------|--------|
| Facility properties | `name`, `display_name`, `base_url` |
| Facility metadata | `info()` |
| Resource listing | `resources()`, `resource(name)`, `resource_by_id(id)` |
| Resource metadata | `Resource.description`, `Resource.group` |
| Job listing | `Resource.jobs()` |
| Incident status | `incidents()`, `incident(id)`, `events(incident_id)` |

## 1 · Authenticate with Globus

Before running this notebook, obtain (or refresh) an ALCF Globus token by
running the following command **in a terminal** from the repository root:

```bash
uv run scripts/tokens/get_globus_token.py --facilities alcf
```

It opens a browser window — paste the authorization code back when prompted.
On subsequent runs the saved refresh token is used automatically and no
browser step is needed.

Once the command completes, return here and run the cells below.

## 2 · Load token and build a refresh-aware provider

In [ ]:
import json
from pathlib import Path
import globus_sdk

# Must match the CLIENT_ID and scope in get_globus_token.py
CLIENT_ID   = "8b84fc2d-49e9-49ea-b54d-b3a29a70cf31"
ALCF_SCOPE  = (
    "https://auth.globus.org/scopes/"
    "6be511f6-a071-471f-9bc0-02a0d0836723/filesystem"
)
TOKEN_FILE  = Path.home() / ".globus" / "auth_tokens.json"


def _load_alcf_token_data() -> dict:
    """Return the ALCF token entry from the saved Globus token file."""
    if not TOKEN_FILE.exists():
        raise FileNotFoundError(
            f"{TOKEN_FILE} not found — run the authentication cell above."
        )
    tokens = json.loads(TOKEN_FILE.read_text())
    for tok in tokens.get("other_tokens", []):
        if ALCF_SCOPE in tok.get("scope", ""):
            return tok
    raise RuntimeError(
        "No ALCF token found in token file. "
        "Re-run the authentication cell with --facilities alcf."
    )


def make_alcf_token_provider():
    """
    Return a zero-argument callable that always produces a valid ALCF
    access token, using the saved Globus refresh token when possible.

    FacilityClient calls this automatically on 401/403 responses.
    """
    globus_client = globus_sdk.NativeAppAuthClient(CLIENT_ID)

    def _get_token() -> str:
        tok_data = _load_alcf_token_data()
        refresh_token = tok_data.get("refresh_token")
        if not refresh_token:
            return tok_data["access_token"]
        resp = globus_client.oauth2_refresh_token(refresh_token)
        return resp.data["access_token"]

    return _get_token


# Load current access token for the initial connection
alcf_token_data    = _load_alcf_token_data()
alcf_access_token  = alcf_token_data["access_token"]
alcf_token_provider = make_alcf_token_provider()

print(f"ALCF token loaded (first 12 chars): {alcf_access_token[:12]}...")

## 3 · Create the FacilityClient

In [ ]:
from amscrot.client.client import Client

# From amsc_client.facility.config.ALCF_CONFIG
ALCF_ENDPOINT     = "https://api.alcf.anl.gov"
ALCF_DISPLAY_NAME = "Argonne Leadership Computing Facility"

client = Client()
alcf = client.facility(
    ALCF_ENDPOINT,
    token=alcf_access_token,
    token_provider=alcf_token_provider,
    name=ALCF_DISPLAY_NAME,
)

print(alcf)

## 4 · Facility properties

`name`, `display_name`, and `base_url` mirror the `amsc_client` interface.

In [ ]:
print(f"name:         {alcf.name}")
print(f"display_name: {alcf.display_name}")
print(f"base_url:     {alcf.base_url}")

## 5 · Facility info

`info()` makes a live call to the IRI `/facility` endpoint and returns the
native `amsc_iri` facility object.

In [ ]:
info = alcf.info()
print(info)

## 6 · Resources

Discovery is cached on first call.  `resources()` returns compute, storage,
and network resources; incidents and other types are filtered out automatically.

In [ ]:
resources = alcf.resources()
print(f"{len(resources)} resource(s) found:\n")
for r in resources:
    print(f"  {r.id}")
    print(f"    name:        {r.name}")
    print(f"    type:        {r.resource_type}")
    print(f"    status:      {r.status}")
    print(f"    description: {r.description or '(none)'}")
    print(f"    group:       {r.group or '(none)'}")
    print()

### Look up a resource by name (case-insensitive)

In [ ]:
# Adjust the name to match what the ALCF endpoint returns above
RESOURCE_NAME = "Polaris"

try:
    polaris = alcf.resource(RESOURCE_NAME)
    print(polaris)
except ValueError as exc:
    print(f"Not found: {exc}")

### Look up a resource by UUID (live API call, not cached)

In [ ]:
if resources:
    resource_id = resources[0].id
    r = alcf.resource_by_id(resource_id)
    print(f"resource_by_id({resource_id!r}):")
    print(f"  name:   {r.name}")
    print(f"  status: {r.status}")
else:
    print("No resources available — skipping resource_by_id demo.")

## 7 · Job listing

`resource.jobs()` fetches the current job list for a resource (live call,
not cached). Each returned `Job` supports `refresh()`, `wait()`, and
`cancel()`.

In [ ]:
if resources:
    target = resources[0]
    print(f"Fetching jobs for '{target.name}'...")
    jobs = target.jobs()
    print(f"{len(jobs)} job(s) found.")
    for job in jobs[:5]:          # show at most 5
        print(f"  {job.id}  state={job.state}")
    if len(jobs) > 5:
        print(f"  ... and {len(jobs) - 5} more.")
else:
    print("No resources available — skipping jobs demo.")

## 8 · Incident status

All three methods are live API calls (never cached).
The returned objects are native `amsc_iri.models.Incident` /
`amsc_iri.models.Event` instances — introspect them directly.

In [ ]:
incidents = alcf.incidents()
print(f"{len(incidents)} incident(s) at ALCF.")
for inc in incidents:
    print(f"  {inc}")

In [ ]:
if incidents:
    inc_id = incidents[0].id
    print(f"Fetching incident {inc_id!r}...")
    inc = alcf.incident(inc_id)
    print(inc)

    print(f"\nEvents for incident {inc_id!r}:")
    events = alcf.events(inc_id)
    print(f"  {len(events)} event(s).")
    for evt in events:
        print(f"  {evt}")
else:
    print("No active incidents — skipping incident/events demo.")

## 9 · (Optional) Submit a test job

Uncomment and adjust the cell below to submit a real job.
Set `SUBMIT = True` to enable it.

In [ ]:
SUBMIT = False  # set to True to actually submit

if SUBMIT and resources:
    compute = next((r for r in resources if r.resource_type == "compute"), None)
    if compute is None:
        print("No compute resource found — cannot submit.")
    else:
        job = compute.submit(
            executable="/bin/echo",
            arguments=["hello from amscrot FacilityClient"],
            nodes=1,
            queue="debug",
            account="",          # <-- set your allocation account
            duration=60,
            filesystems="home",  # ALCF-specific custom attribute
        )
        print(f"Submitted: {job}")
        print("Waiting for completion...")
        job.wait(timeout=300, poll_interval=10)
        print(f"Done — state={job.state}, exit_code={job.exit_code}")
else:
    print("Submission skipped (SUBMIT=False or no resources).")